# Multi frame prediction and saving new annotations in COCO format

### Prediction post-processing and evaluation

1. Generate the pixel-wise prediction (`img_pred`) and save
   `{fnam}_prediction.png`.

2. Label connected foreground regions:
   `new_labels = label(img_pred > 0)`.

3. Using the unexpanded `new_labels`, extract region IDs and geometric
   properties, and assign each region a cell type by majority vote
   over non-background prediction pixels (`df['gray']`, `df['anatomy']).
   Geometric properties are supplementary outputs and are not used
   to calculate segmentation scores.

4. Map the assigned grayscale values back onto `new_labels` and save
   `{fnam}_pred_cleaned.png`.

5. Export the reconstructed mask as `{fnam}_annot_coco.json`.

6. Remap the ground-truth grayscale values and convert both the GT
   and reconstructed prediction to class IDs. Border-touching cells
   are retained in both masks.

7. Calculate IoU, Dice and Accuracy, and save `evaluation_scores.csv`.

`expanded_labels` is generated separately for supplementary
visualisation. It is not used for class assignment, geometric
property extraction, mask reconstruction or score calculation.

In [ ]:
import torch
import cv2
import os
import numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import time
import itertools
import pandas as pd

from skimage import data, graph
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import closing, square
from skimage.color import label2rgb
from skimage.segmentation import expand_labels

from src.load_mrcnn import get_model, load_checkpoint, predict_multi_frame

### Read prediction model

In [ ]:
# GPU
GPU_ID = 0
DEVICE = torch.device(
    f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu"
)

# Configuration: select matching patch size and checkpoint
image_size = 512
IMAGE_SIZE = [image_size, image_size]

MODEL_FILE = "512_thin_multispecies_700_epoch.tar"
CHECKPOINT_FILE = os.path.join("./model", MODEL_FILE)
MODEL_LABEL = MODEL_FILE

# Optional test-patch directory: use the existing dataset path
TEST_DIR = f"dataset/png_{image_size}/image/test"

LEARNING_RATE = 1e-5
WEIGHT_DECAY = 5e-4

# Build and load the MS model
model = get_model().to(DEVICE)

optimizer = torch.optim.AdamW(
    params=model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location=DEVICE,
    weights_only=True
)

load_checkpoint(
    checkpoint, model, optimizer, LEARNING_RATE
)

model.eval()

print(f"Device: {DEVICE}")
print(f"Loaded MS checkpoint: {CHECKPOINT_FILE}")

## Prediction

In [ ]:
model.train(False)

img_num = 1

testfilename = [fl for fl in os.listdir(TEST_DIR) if fl.endswith('.png')]
print('You have ', str(len(testfilename)-1), \
      ' test images, and you selected image No. ', str(img_num))
img0 = Image.open(os.path.join(TEST_DIR,testfilename[img_num])).convert('RGB')


Check model using test image !

In [ ]:
img = cv2.imread(os.path.join(TEST_DIR,testfilename[img_num]))
res2 = predict_multi_frame(model, img, IMAGE_SIZE, 0.7, 0.7, DEVICE)
plt.imshow(res2)
plt.show()

Predictions are generated from overlapping patches. Only the central
384 × 384 pixels of each 512 × 512 patch, or 192 × 192 pixels of each, 256 × 256 patch, are retained and stitched together to reconstruct the full image.

In [ ]:

sp_to_filename = {
    "acapod": "acapod2_01_img.png",
    "alllit": "alllit2_01_img.png",
    "burspi": "burspi1_01_img.png",
    "euclon": "euclon1_01_img.png",
    "exostr": "exostr8_01_img.png",
    "hakdac": "hakdac2_01_img.png"
}

input_dir = 'dataset/performance_check/image'
gt_dir = 'dataset/performance_check/mask'
base_dir = './inference_results_recheck20261011'


# parameter setting according to image size
if image_size == 512:
    pad = 64
    sz = 384
    l = list(range(4))
elif image_size == 256:
    pad = 32
    sz = 192
    l = list(range(8))
else:
    raise ValueError(f"Unsupported image_size: {image_size}")



predictions = []
label_results = []


# inference loop
for sp, filename in sp_to_filename.items():
    data_img = os.path.join(input_dir, filename)
    print(f"Processing {data_img} ...")

    img_in = cv2.imread(data_img)
    if img_in is None:
        print(f"error: {data_img}")
        continue

    img_padded = cv2.copyMakeBorder(img_in, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    height, width, _ = img_padded.shape
    img_pred = np.zeros_like(img_in[:, :, 0])

    start = time.time()

    for i, j in tqdm(itertools.product(l, l), desc=f"Inference on {filename}"):
        img_patch = img_padded[sz*i:sz*i+image_size, sz*j:sz*j+image_size, :]
        res2 = predict_multi_frame(model, img_patch, (image_size, image_size), 0.7, 0.9, DEVICE)
        res2 = res2[pad:pad+sz, pad:pad+sz]
        img_pred[sz*i:sz*i+sz, sz*j:sz*j+sz] = res2

    end = time.time()

    fnam = os.path.splitext(filename)[0]
    raw_dir = os.path.join(base_dir, MODEL_FILE, 'prediction_raw')
    os.makedirs(raw_dir, exist_ok=True)

    # save image
    integ_img = Image.fromarray(img_pred)
    integ_img.save(os.path.join(raw_dir, fnam + '_prediction.png'))

    new_labels = label(img_pred > 0)
    expanded_labels = expand_labels(new_labels, distance=15)
    
    label_results.append((fnam, new_labels, expanded_labels))


    print(f"{fnam}: inference completed (GPU time = {end - start:.2f} sec)")
    predictions.append((fnam, img_pred))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, (fnam, pred_img) in zip(axes, predictions):
    ax.imshow(pred_img, cmap='gray')
    ax.set_title(fnam)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Relabelling the reconstructed prediction image

Connected foreground regions are labelled using
`label(img_pred > 0)`. Regions touching the image border are retained.

The original, unexpanded labels (`new_labels`) are used for
supplementary geometric property extraction, majority-vote cell-type
assignment and prediction mask reconstruction.

Expanded labels (`expanded_labels`) are generated only for
supplementary visualisation. No border removal or graph construction
is performed in this notebook.


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(24, 8)) 
axes = axes.flatten()

for idx, (fnam, new_labels, expanded_labels) in enumerate(label_results):
    ax1 = axes[2 * idx]
    ax2 = axes[2 * idx + 1]

    ax1.imshow(new_labels)
    ax1.set_title(f"{fnam} - Original")
    ax1.axis('off')

    ax2.imshow(expanded_labels)
    ax2.set_title(f"{fnam} - Expanded")
    ax2.axis('off')

plt.tight_layout()
plt.show()


Predictive maps of cell types were labelled and the morphological elements were obtained from them using the reagionalprops function of SKIMAGE, and then stored in the panda data.

In [ ]:
# Extract region IDs and supplementary geometric properties
# from the unexpanded predicted regions.
# Geometric properties (area, centroid, shape, etc.) are not used
# to calculate IoU, Dice or Accuracy.
# Region IDs and majority-vote class assignments are required
# for subsequent mask reconstruction.
# Border-touching regions are retained; clear_border is not applied.
# Boundary expansion for anatomical measurements is reserved
# for a separate workflow after manual review/correction.

from skimage.measure import regionprops_table
import pandas as pd

all_dfs = []

for (fnam, pred_img), (_, new_labels, _) in zip(predictions, label_results):
    print(f"Processing props for {fnam}")


    props = regionprops_table(
        new_labels,
        properties=('label', 'centroid', 'bbox', 'orientation',
                    'axis_major_length', 'axis_minor_length', 'area'),
    )
    df = pd.DataFrame(props)


    def get_gray(p):
        output = pred_img[new_labels == p]
        out = output[np.isin(output, [64, 128, 192, 255])]
        if len(out) == 0:
            return 0
        return np.argmax(np.bincount(out))

    t1 = time.time()
    label_list = df['label'].to_numpy()
    gray_values = [get_gray(lbl) for lbl in label_list]
    df['gray'] = gray_values
    t2 = time.time()
    print(f"{fnam} : {t2 - t1:.2f} sec")


    color = [(0, 0, 0), (1, 159/255, 155/255), (144/255, 229/255, 161/255),
             (161/255, 201/255, 244/255), (1, 180/255, 130/255)]
    cell_type = ['stuff', 'vessel', 'ray', 'fiber', 'parenchyma']
    gray = [0, 64, 128, 192, 255]

    gr_cell_dict = dict(zip(gray, cell_type))
    gr_c_dict = dict(zip(gray, color))


    df['anatomy'] = df['gray'].map(gr_cell_dict)
    df['color'] = df['gray'].map(gr_c_dict)
    df['image'] = fnam


    os.makedirs(os.path.join(raw_dir, "cell_props"), exist_ok=True)
    df.to_csv(os.path.join(raw_dir,"cell_props", f"{fnam}_cell_props.csv"), index=False)
    

    all_dfs.append(df)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv("all_cell_props_combined.csv", index=False)



Each original, unexpanded labelled region is assigned the most frequent non-background prediction class within that region. Background pixels are excluded from the vote.

link-table
|            | vessel | ray  | fiber | parenchyma |  
| ---------- | ------ | ---- | ----- | ---------- |
| vessel     | 1      | 2    | 3     | 2          |
| ray        | 2      | 4    | 6     | 4          |
| fiber      | 3      | 6    | 9     | 6          |
| parenchyma | 2      | 4    | 6     | 4          |



| cell_type | 'stuff' | 'vessel'             | 'ray'                     | 'fiber'                    | 'parenchyma'        |
| --------- | ------- | -------------------- | ------------------------- | -------------------------- | ------------------- |
| color     | (0,0,0) | （1,159/255,155/255) | (144/255,229/255,161/255) | (161/255,201/255,244/255), | (1,180/255,130/255) |
| gray      | 0       | 64                   | 128                       | 192                        | 255                 |

***

Map anatomy data on the new labels.

In [ ]:
import copy
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

annot_save_dir = os.path.join(base_dir, MODEL_FILE, 'prediction_cleaned')
os.makedirs(annot_save_dir, exist_ok=True)

annotated_images = []

for (fnam, new_labels, _), df in zip(label_results, all_dfs):
    print(f"Generating anatomical annotation (unexpanded) for: {fnam}")
    
    new_labels_copy = copy.deepcopy(new_labels).flatten()
    new_annotation = np.zeros_like(new_labels_copy, dtype=np.uint8)

    replace_func = {
        int(label): int(gray) for label, gray in zip(df['label'].values, df['gray'].values)
    }

    t1 = time.time()
    replace_result = np.array(list(map(replace_func.get, new_labels_copy)))
    result_index = np.where(replace_result != None)[0]
    new_annotation[result_index] = replace_result[result_index]
    t2 = time.time()
    print(f"{fnam} : annotation completed in {t2 - t1:.2f} sec")

    newly_annotated_img = new_annotation.reshape(new_labels.shape)

    out_path = os.path.join(annot_save_dir, f"{fnam}_pred_cleaned.png")
    Image.fromarray(newly_annotated_img).save(out_path)

    annotated_images.append((fnam, newly_annotated_img))



In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (fnam, img) in zip(axes, annotated_images):
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{fnam} (original labels)")
    ax.axis('off')

plt.tight_layout()
plt.show()


Check new labels 

In [ ]:
from src.create_annotations import *

# original 2560 x 2560 image 
#data_img = 'dataset/original/Alllit_test_original.png'
#data_img = 'dataset/original/kusunoki.png'


# Label ids of the dataset
category_ids = {
    "stuff" : 0,
    "vessel": 1,
    "ray": 2,
    "fiber": 3,
    "parenchyma": 4
}

# Define which colors match which categories in the images
category_colors = {
    "(0, 0, 0)" : 0, # stuff
    "(64, 64, 64)": 1, # vessel
    "(128, 128, 128)": 2, # ray
    "(192, 192, 192)": 3, # fiber
    "(255, 255, 255)": 4 # parenchyma
}
"""
# Label ids of the dataset
category_ids = {
    "stuff" : 0,
    "unknown": 1,
}

# Define which colors match which categories in the images
category_colors = {
    "(0, 0, 0)" : 0, # stuff
    "(64, 64, 64)": 1 # unknown
}
"""

In [ ]:
import json
from pathlib import Path

multipolygon_ids = []

annot_json_save_dir = os.path.join(base_dir, MODEL_FILE, 'prediction_cleaned_json')
os.makedirs(annot_json_save_dir, exist_ok=True)

for (fnam, new_labels, _), df in zip(label_results, all_dfs):
    print(f"Generating COCO-style JSON for: {fnam}")
    
    label_map = np.zeros_like(new_labels, dtype=np.uint8)
    for lbl, gray in zip(df['label'], df['gray']):
        label_map[new_labels == lbl] = gray

    data_img = os.path.join(input_dir, fnam + ".png")

    coco_format = get_coco_json_format()
    coco_format["info"] = info()
    coco_format["categories"] = create_category_annotation(category_ids)

    coco_format["images"], coco_format["annotations"], annotation_cnt = \
        singleshot_annotations_info(label_map, data_img, category_ids, category_colors, multipolygon_ids)

    json_path = os.path.join(annot_json_save_dir, f"{fnam}_annot_coco.json")
    with open(json_path, "w") as outfile:
        json.dump(coco_format, outfile)

    print(f"{fnam}: Created {annotation_cnt} annotations → {json_path}")


***

# Validation


1. Load the ground-truth mask in grayscale.

2. Remap its grayscale values using `gt_remap_dict` and
   `remap_gt_gray` to match the prediction labels.

3. Convert the ground-truth mask to class IDs (0–4) using
   `normalize_mask`.

4. Load the reconstructed prediction mask (`*_pred_cleaned.png`)
   and convert it to class IDs using the same mapping.

5. Retain cells touching the image border in both masks.
   No `clear_border` operation is applied.

6. Calculate:
   - Per-class IoU and Dice for the four non-background classes,
     together with their unweighted means.
   - Accuracy over pixels where the ground-truth class is non-background.

7. Display the input image, ground-truth mask, prediction mask
   and calculated scores, and save the comparison figure.

8. Save the scores for all evaluated images to `evaluation_scores.csv`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.measure import label
import os


label_mapping = {0: 0, 64: 1, 128: 2, 192: 3, 255: 4}
num_classes = len(label_mapping)
ignore_class = 0  # stuff を無視


def normalize_mask(mask):
    return np.vectorize(label_mapping.get)(mask)

def remap_gt_gray(mask, remap_dict):
    remapped = np.copy(mask)
    for src, dst in remap_dict.items():
        remapped[mask == src] = dst
    return remapped

def compute_iou(gt, pred, num_classes, ignore_class=None):
    iou_list = []
    for cls in range(num_classes):
        if cls == ignore_class:
            continue
        gt_cls = (gt == cls)
        pred_cls = (pred == cls)
        intersection = np.logical_and(gt_cls, pred_cls).sum()
        union = np.logical_or(gt_cls, pred_cls).sum()
        if union == 0:
            continue
        iou_list.append(intersection / union)
    return np.mean(iou_list), iou_list

def compute_dice(gt, pred, num_classes, ignore_class=None):
    dice_list = []
    for cls in range(num_classes):
        if cls == ignore_class:
            continue
        gt_cls = (gt == cls)
        pred_cls = (pred == cls)
        tp = np.logical_and(gt_cls, pred_cls).sum()
        denom = gt_cls.sum() + pred_cls.sum()
        if denom == 0:
            continue
        dice_list.append(2 * tp / denom)
    return np.mean(dice_list), dice_list

def compute_accuracy(gt, pred, ignore_class=None):
    if ignore_class is None:
        return (gt == pred).sum() / gt.size
    mask = gt != ignore_class
    return (gt[mask] == pred[mask]).sum() / mask.sum()



In [ ]:
id_to_classname = {v: k for k, v in category_ids.items()}


gt_dir = 'dataset/performance_check/mask'
score_dir = os.path.join(base_dir, MODEL_FILE, 'scores')
vis_dir = os.path.join(base_dir, MODEL_FILE, 'visual_comparison')
os.makedirs(score_dir, exist_ok=True)
os.makedirs(vis_dir, exist_ok=True)


score_records = []


for idx, (sp, filename) in enumerate(sp_to_filename.items()):
    fnam = os.path.splitext(filename)[0]
    fnam_short = fnam[:-4]

    img_path = os.path.join(input_dir, filename)
    gt_path = os.path.join(gt_dir, fnam_short + "_msk.png")
    pred_dir = os.path.join(base_dir, MODEL_FILE, 'prediction_cleaned')
    pred_path = os.path.join(pred_dir, fnam + "_pred_cleaned.png")


    img = np.array(Image.open(img_path).convert("RGB"))
    gt_raw = np.array(Image.open(gt_path).convert("L"))
    pred_raw = np.array(Image.open(pred_path).convert("L"))


    gt_remap_dict = {0: 64, 64: 128, 128: 192, 192: 255, 255: 0}
    gt_remapped = remap_gt_gray(gt_raw, gt_remap_dict)
    gt_norm = normalize_mask(gt_remapped)

    # normalise inference results
    pred_norm = normalize_mask(pred_raw)

    # calculate scores
    mean_iou, per_class_iou = compute_iou(gt_norm, pred_norm, num_classes, ignore_class)
    mean_dice, per_class_dice = compute_dice(gt_norm, pred_norm, num_classes, ignore_class)
    acc = compute_accuracy(gt_norm, pred_norm, ignore_class)


    fig, axes = plt.subplots(1, 4, figsize=(20, 6))
    axes[0].imshow(img)
    axes[0].set_title('Input Image')
    axes[1].imshow(gt_norm, cmap='gray')
    axes[1].set_title('GT Mask')
    axes[2].imshow(pred_norm, cmap='gray')
    axes[2].set_title('Predicted Mask')


    text = f"Mean IoU : {mean_iou:.4f}\nMean Dice: {mean_dice:.4f}\nAccuracy : {acc:.4f}\n"
    for cls_id in range(num_classes):
        if cls_id == ignore_class:
            continue
        cls_name = id_to_classname.get(cls_id, f"Class {cls_id}")
        text += f"\n{cls_name} IoU : {per_class_iou[cls_id - 1]:.4f}"
        text += f"\n{cls_name} Dice: {per_class_dice[cls_id - 1]:.4f}"
    axes[3].axis('off')
    axes[3].text(0, 0.5, text, fontsize=12, va='center')

    plt.tight_layout()
    fig.savefig(os.path.join(vis_dir, f"{fnam}_comparison.png"), dpi=300)
    plt.show()
    plt.close(fig)


    record = {
        "filename": fnam,
        "mean_iou": mean_iou,
        "mean_dice": mean_dice,
        "accuracy": acc,
    }
    for cls_id in range(num_classes):
        if cls_id == ignore_class:
            continue
        cls_name = id_to_classname.get(cls_id, f"class{cls_id}")
        record[f"{cls_name}_iou"] = per_class_iou[cls_id - 1]
        record[f"{cls_name}_dice"] = per_class_dice[cls_id - 1]
    score_records.append(record)


df_scores = pd.DataFrame(score_records)
df_scores.to_csv(os.path.join(score_dir, "evaluation_scores.csv"), index=False)